# SOLiD-PINN v0.10.1b — Native PyBaMM / IDAKLU (current-conservation revision)
CPU · Run all · No Drive or token. Four regression cases, 12 sequential stages in smoke tests. If successful, 45 benchmark stages and 36 comparisons. Native tolerances tightened 100x; physical gates unchanged. No fallback solver if PyBaMM fails.


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess,sys,os,tempfile,json,hashlib
REV="7acbb879693e0357a231e7f89740e7bd2e7f584a"
repo=Path(tempfile.mkdtemp(prefix="solid-pybamm-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
blobs={"canonical_fv.py":"102156ad4699a86114296272088bb5e13d163928","foundation_fem.py":"c41c62abed072efc699e037cd4e00b559b99425c","verify_foundation.py":"9de814b7f6b22fe86f9e56c0af76671aa30a6daf","idas_bridge.py":"54d8135540f3bdab8ad53792b2f5467837ec9365","foundation_pybamm.py":"8d0df57101e979b79630498798dd9592eace440d","test_foundation_pybamm.py":"607ebaeadda5ae9783e6146c9261f223238b1399"}
for n,h in blobs.items(): assert subprocess.check_output(["git","hash-object",str(repo/"scripts"/n)],text=True).strip()==h
hashes={n:hashlib.sha256((repo/"scripts"/n).read_bytes()).hexdigest() for n in blobs}
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",PYBAMM_DISABLE_TELEMETRY="true")
subprocess.run([sys.executable,"-m","pip","install","pybamm==25.12.1","pybammsolvers==0.3.3","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)


## Execute benchmark and preserve failures


In [ ]:
out=Path("/content/foundation_pybamm_checkpoint.json")
smoke=subprocess.run([sys.executable,"-m","pytest","-q","-s","scripts/test_foundation_pybamm.py"],cwd=repo,env=env)
if smoke.returncode==0:
    run=subprocess.run([sys.executable,"-u","scripts/foundation_pybamm.py","--out",str(out)],cwd=repo,env=env)
    d=json.loads(out.read_text()) if out.exists() else {"complete":False,"errors":["No benchmark checkpoint"]}
    d["runner_returncode"]=run.returncode
else:
    d={"complete":False,"errors":["PyBaMM smoke test failed; inspect stdout"],"rows":[],"pairs":[]}
d.update(source_revision=REV,source_hashes=hashes,source_git_blobs=blobs,targeted_tests_passed=4 if smoke.returncode==0 else 0)
if d["complete"]:
    assert len(d["rows"])==45 and len(d["pairs"])==36
    assert d["comparison_gate"]==all(x["max_voltage_mv"]<=1 and x["capacity_delta_pct"]<=.5 for x in d["pairs"])
print({k:v for k,v in d.items() if k.endswith("_gate") or k=="complete"})


## Export evidence


In [ ]:
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
dest=Path("/content")/f"SOLiD-PINN_v0101b_colab_{sha[:20]}.json"
dest.write_bytes(raw)
assert dest.read_bytes()==raw
print(dest,sha)
files.download(str(dest))
